# Paper 4 — 04 · Gemma Scope SAE feature analysis (H1e)

**Gemma anchor only — corroboration.** Load pretrained Gemma Scope JumpReLU residual SAEs (no training), identify detection features (separate harm_en vs benign_en) and refusal features (separate refusal vs compliance), and compare firing on EN vs RO harmful prompts across the bands. H1e: detection features under-fire on RO in the detection band.

**Output:** `results/gemma-2-2b/sae_features.json`.

In [ ]:
%%capture
# Pinned to requirements.txt. Wheel-only on A100 / CUDA 12; restart rarely needed.
!pip install -U \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    'datasets>=3.0' \
    'scikit-learn>=1.4' \
    'transformer-lens>=2.9' \
    'sae-lens>=4.0' \
    python-dotenv requests huggingface_hub ipywidgets pyyaml matplotlib seaborn -q


In [ ]:
import os, json, gc, sys, hashlib
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Secrets (Colab) -> env, so the Paper 2 judge + gated HF models work
#     end-to-end with no manual steps. Set these in Colab -> Secrets first. ---
try:
    from google.colab import userdata
    for _k in ("OPENROUTER_API_KEY", "HF_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v:
                os.environ[_k] = _v
        except Exception:
            print(f"[secrets] {_k} not set in Colab Secrets — add it if a cell needs it.")
except Exception:
    pass
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(os.environ["HF_TOKEN"], add_to_git_credential=False)

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper4-interpretability")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")
PAPER3_ROOT = Path("/content/drive/MyDrive/PhD/paper3-alignment")

DATA_DIR     = DRIVE_ROOT / "data"
CONTRAST_DIR = DATA_DIR / "contrastive"
ACT_DIR      = DATA_DIR / "activations"
PROBE_DIR    = DATA_DIR / "probes"
SPLITS_DIR   = DATA_DIR / "splits"
RESULTS_DIR  = DRIVE_ROOT / "results"
FIG_DIR      = DRIVE_ROOT / "figures"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [CONTRAST_DIR, ACT_DIR, PROBE_DIR, SPLITS_DIR, RESULTS_DIR, FIG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Reuse Paper 2 judge harness + Paper 3 helpers; Paper 4 src/ ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))      # judges.py, llm_judge.py
sys.path.insert(0, str(DRIVE_ROOT / "src"))        # paths, capture, probes, patching, sae_utils, contrastive, behavioral

# --- A100 sanity ---
assert torch.cuda.is_available(), "Need a GPU runtime (A100 high-RAM)."
torch.backends.cuda.matmul.allow_tf32 = True
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## Configuration

In [ ]:
# --- Anchor selection. Re-run the notebook once per anchor. ---
# SAE anchor (H1e available):  google/gemma-2-2b-it
# Cross-arch anchors:          Qwen/Qwen2.5-3B-Instruct, meta-llama/Llama-3.2-3B-Instruct
ANCHOR = "google/gemma-2-2b-it"

from paths import short_of, family_of
short  = short_of(ANCHOR)
family = family_of(ANCHOR)
print(f"ANCHOR : {ANCHOR}\nfamily : {family}\nshort  : {short}")


In [ ]:
assert short == 'gemma-2-2b', 'H1e is Gemma-only (Gemma Scope SAEs).'

## 1. Load cells + behavioral labels + bands; load anchor

In [ ]:
out = CONTRAST_DIR / short
def _read(n): return [json.loads(l) for l in (out/f'{n}.jsonl').read_text().splitlines() if l.strip()]
cells = {n: _read(n) for n in ['harm_en','benign_en','harm_ro','benign_ro']}
beh = {json.loads(l)['id']: json.loads(l)['label'] for l in (out/'behavioral_labels.jsonl').read_text().splitlines() if l.strip()}
bands = json.loads((RESULTS_DIR / short / 'bands.json').read_text())
band_layers = sorted(set(bands['detection'] + bands['execution']))
from transformers import AutoModelForCausalLM, AutoTokenizer
from capture import capture_assistant_prefix
tok = AutoTokenizer.from_pretrained(ANCHOR); tok.padding_side='left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(ANCHOR, torch_dtype=torch.bfloat16, device_map='cuda').eval()

## 2. Capture residuals per cell (reuse nb02 cache if present)

In [ ]:
def cap(name):
    c = ACT_DIR / short / f'{name}.pt'
    if c.exists(): return torch.load(c)
    c.parent.mkdir(parents=True, exist_ok=True)
    a = capture_assistant_prefix(model, tok, [r['text'] for r in cells[name]]); torch.save(a, c); return a
acts = {n: cap(n) for n in cells}
print({n: tuple(a.shape) for n, a in acts.items()})

## 3. Per-band-layer: load SAE, find detection + refusal features, compare EN vs RO firing

Detection features separate harm_en vs benign_en; refusal features separate
behaviorally-refused vs complied prompts. H1e: detection features under-fire
on RO in the detection band; refusal features fire comparably.

In [ ]:
from sae_utils import load_gemma_scope_sae, encode_acts, difference_in_means_features, en_ro_firing_gap
WIDTH = '16k'   # ablate '65k' in a second pass (plan §8)
def beh_mask(names, label):
    rows = [r for n in names for r in cells[n]]
    return np.array([beh.get(r['id'])==label for r in rows])
rows_en = cells['harm_en'] + cells['benign_en']
ref_mask = np.array([beh.get(r['id'])=='refuse' for r in rows_en])
per_layer = []
for L in band_layers:
    sae = load_gemma_scope_sae(L, width=WIDTH)
    z = {n: encode_acts(sae, acts[n][:, L]) for n in cells}
    z_en = np.concatenate([z['harm_en'], z['benign_en']], 0)
    det_feats = difference_in_means_features(z['harm_en'], z['benign_en'])
    ref_feats = difference_in_means_features(z_en[ref_mask], z_en[~ref_mask])
    gap_det = en_ro_firing_gap(z['harm_en'], z['harm_ro'], det_feats)
    gap_ref = en_ro_firing_gap(z['harm_en'], z['harm_ro'], ref_feats)
    band = 'detection' if L in bands['detection'] else 'execution'
    per_layer.append({'layer': L, 'band': band,
        'det_en_minus_ro': gap_det['en_minus_ro'], 'ref_en_minus_ro': gap_ref['en_minus_ro']})
    del sae; gc.collect(); torch.cuda.empty_cache()
    print(f"L{L} ({band}): det EN-RO firing {gap_det['en_minus_ro']:+.3f} | ref {gap_ref['en_minus_ro']:+.3f}")

## 4. Save + plot

In [ ]:
rs = RESULTS_DIR / short; rs.mkdir(parents=True, exist_ok=True)
(rs / 'sae_features.json').write_text(json.dumps({'anchor_model': ANCHOR, 'short': short,
    'analysis': 'sae_features', 'width': WIDTH, 'bands': bands, 'per_layer': per_layer}, indent=2))
import matplotlib.pyplot as plt
L = [p['layer'] for p in per_layer]
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(L, [p['det_en_minus_ro'] for p in per_layer], 'o-', label='detection features (EN-RO firing)')
ax.plot(L, [p['ref_en_minus_ro'] for p in per_layer], 's-', label='refusal features (EN-RO firing)')
ax.axhline(0, color='k', lw=0.5)
for b in bands['detection']: ax.axvspan(b-0.5, b+0.5, color='C0', alpha=0.06)
ax.set_xlabel('layer'); ax.set_ylabel('EN minus RO firing rate'); ax.legend(fontsize=8)
ax.set_title(f'{short}: SAE feature firing (H1e)')
fig.tight_layout(); fig.savefig(FIG_DIR / f'sae_firing_{short}.pdf'); plt.show()